<h5> Imports de librerías de pyspark </h5>

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import *
from typing import Dict, List
from utils.functions.prepare_system_columns import prepare_system_dataframe
from utils.functions.upsert import upsert_data
from utils.functions.validate_natural_key import validate_natural_key_df

In [0]:
input_catalog = "silver"
input_table_name = "total_sales"
output_catalog = "gold"
schema = "sales"
gold_schema = "sales_summarized"
table_name = f"{output_catalog}.{gold_schema}.customer_segmentation"
force_history = False

<h5> Configuración de la aplicación de Spark </h5>

In [0]:
spark = (
    SparkSession.builder
    .appName('SalesPipelineGoldApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In [0]:
sales_silver_df = spark.table(f"{input_catalog}.{schema}.{input_table_name}")

In [0]:
from pyspark.sql.functions import (
    col, max as spark_max, min as spark_min, count, sum as spark_sum, 
    avg, datediff, current_date, when, lit, round as spark_round,
    ntile, concat_ws
)

from pyspark.sql.window import Window

# Calcular fecha de referencia (última fecha en los datos)
max_date_row = sales_silver_df.select(spark_max(col("FECHA")).alias("fecha_maxima")).collect()
reference_date = max_date_row[0]["fecha_maxima"] if max_date_row else current_date()


# Análisis RFM por cliente
rfm_analysis = (
    sales_silver_df
    .groupBy("IDCLIENTE", "NOMBRE_Y_APELLIDO", "EDAD", "PROVICIA_CLIENTE", "LOCALIDAD_CLIENTE")
    .agg(
        # Recency: días desde la última compra
        datediff(lit(reference_date), spark_max("FECHA")).alias("dias_desde_ultima_compra"),
        
        # Frequency: cantidad de compras
        count("IDVENTA").alias("frecuencia"),
        
        # Monetary: valor total gastado
        spark_sum(col("PRECIO") * col("CANTIDAD")).alias("valor_monetario"),
        
        # Métricas adicionales
        avg(col("PRECIO") * col("CANTIDAD")).alias("valor_promedio_compra"),
        spark_min("FECHA").alias("fecha_primera_compra"),
        spark_max("FECHA").alias("fecha_ultima_compra"),
        count("IDPRODUCTO").alias("total_productos_comprados")
    )
)

# Calcular percentiles para segmentación RFM
window_spec = Window.orderBy(col("dias_desde_ultima_compra"))
rfm_with_scores = (
    rfm_analysis
    # Recency Score (invertido: menor días = mejor score)
    .withColumn("puntaje_r", 6 - ntile(5).over(Window.orderBy(col("dias_desde_ultima_compra"))))
    # Frequency Score
    .withColumn("puntaje_f", ntile(5).over(Window.orderBy(col("frecuencia"))))
    # Monetary Score
    .withColumn("puntaje_m", ntile(5).over(Window.orderBy(col("valor_monetario"))))
)

# Crear segmentos RFM
rfm_segmented = (
    rfm_with_scores
    .withColumn("puntaje_rfm", concat_ws("-", col("puntaje_r"), col("puntaje_f"), col("puntaje_m")))
    .withColumn(
        "segmento_cliente",
        when((col("puntaje_r") >= 4) & (col("puntaje_f") >= 4) & (col("puntaje_m") >= 4), "Champions")
        .when((col("puntaje_r") >= 3) & (col("puntaje_f") >= 3) & (col("puntaje_m") >= 3), "Loyal Customers")
        .when((col("puntaje_r") >= 4) & (col("puntaje_f") <= 2), "New Customers")
        .when((col("puntaje_r") >= 3) & (col("puntaje_f") <= 2) & (col("puntaje_m") <= 2), "Promising")
        .when((col("puntaje_r") <= 2) & (col("puntaje_f") >= 3) & (col("puntaje_m") >= 3), "At Risk")
        .when((col("puntaje_r") <= 2) & (col("puntaje_f") <= 2), "Lost Customers")
        .when((col("puntaje_m") >= 4), "Big Spenders")
        .otherwise("Regular")
    )
)

# Segmentación por edad
rfm_with_age_segment = (
    rfm_segmented
    .withColumn(
        "segmento_edad",
        when(col("EDAD") < 25, "18-24")
        .when((col("EDAD") >= 25) & (col("EDAD") < 35), "25-34")
        .when((col("EDAD") >= 35) & (col("EDAD") < 45), "35-44")
        .when((col("EDAD") >= 45) & (col("EDAD") < 55), "45-54")
        .when((col("EDAD") >= 55) & (col("EDAD") < 65), "55-64")
        .when(col("EDAD") >= 65, "65+")
        .otherwise("Desconocido")
    )
)

# Calcular días como cliente y tasa de retención
customer_segmentation_final = (
    rfm_with_age_segment
    .withColumn(
        "dias_vida_cliente",
        datediff(col("fecha_ultima_compra"), col("fecha_primera_compra"))
    )
    .withColumn(
        "tasa_frecuencia_compra",
        when(col("dias_vida_cliente") > 0,
             spark_round(col("frecuencia") / (col("dias_vida_cliente") / 30), 2))
        .otherwise(col("frecuencia"))
    )
    .withColumn(
        "activo",
        when(col("dias_desde_ultima_compra") <= 90, lit(True)).otherwise(lit(False))
    )
    .withColumn("CREATED_AT", current_timestamp())
    .withColumn("UPDATED_AT", current_timestamp())
).drop("CREATED_AT", "UPDATED_AT")

customer_segmentation_final.display()

##### Preparamos dataframe para generar tabla en gold
<p>Convertimos todas las columnas a mayúsculas</p> 
<p>Creamos las fechas de sistema</p>
<p>Creamos PK_HASH y R_HASH </p>

In [0]:
transformed_df = prepare_system_dataframe(customer_segmentation_final, ['IDCLIENTE'])

<h4> Carga de datos </h4>

In [0]:
upsert_data(transformed_df,table_name)